# Project: Line Following Unicycle Robot
### Group: 

Idea: implement a unicycle robot that follows a line in a determined region.

Create a world bound, add a line (of type y = ax + b) randomly in the space, place the robot in random coordinates in the bounding box.

Simulation: robot goes towards line, follows it in bounds of world bound.

In [ ]:
import numpy as np
import roboticstoolbox as rtb
import spatialgeometry as sg
import spatialmath as sm
import swift
import time
import matplotlib as mtb
import matplotlib.pyplot as plt


In [13]:
# # environment setup

# env = swift.Swift()
# env.launch(realtime=True,comms="rtc",browser="notebook")
# #env.set_camera_pose([0,0,0])

# # Simulation parameters
# dt = 0.01  # time step
# time = np.arange(0, 10, dt)  # simulation time



In [14]:
# code bits from cat and mouse

# for function definitions
from dataclasses import dataclass
from typing import Sequence, Tuple


In [15]:

mtb.rcParams.update(mtb.rcParamsDefault)

# Import controllers
# from unicycle_controllers import Wanderer, Pursuer

# Visual parameters
FLOOR_HEIGHT = 0.02
COLOR_RED = (0.8, 0.2, 0.3, 1.0)
COLOR_BLUE = (0.2, 0.5, 0.8, 1.0)
COLOR_WHITE = (1.0, 1.0, 1.0, 0.6)


In [16]:
@dataclass
class AgentVisual:
    """Swift geometry representing a single unicycle body."""

    body: sg.Cuboid

    def update(self, base_pose: sm.SE3) -> None:
        self.body.T = base_pose * sm.SE3(0.0, 0.0, 0.05 + FLOOR_HEIGHT)


def make_agent(color: Tuple[float, float, float, float]) -> AgentVisual:
    """Create the visual body for an agent."""
    body = sg.Cuboid(scale=[0.4, 0.2, 0.1], color=color)
    return AgentVisual(body)


def state_to_pose(state: np.ndarray) -> sm.SE3:
    """Convert unicycle state (x, y, theta) to an SE(3) pose."""
    x, y, theta = state
    return sm.SE3(x, y, 0.0) * sm.SE3.Rz(theta)


def add_world_bounds(env: swift.Swift, bounds: Tuple[float, float, float, float]) -> sg.Cuboid:
    """Draw a translucent floor tile highlighting the workspace."""
    min_x, max_x, min_y, max_y = bounds
    width, depth = max_x - min_x, max_y - min_y
    floor = sg.Cuboid(scale=[width, depth, FLOOR_HEIGHT], color=COLOR_WHITE)
    floor.T = sm.SE3((min_x + max_x) / 2.0, (min_y + max_y) / 2.0, 0.01)
    env.add(floor)
    return floor


class UnicycleAgent:
    """Bundle unicycle dynamics and Swift visuals."""

    def __init__(self, name: str, start_state: Sequence[float], color: Tuple[float, float, float, float]) -> None:
        self.name = name
        self._robot = rtb.Unicycle()
        self._robot.init(np.array(start_state, dtype=float))
        self._visual = make_agent(color=color)

    def register(self, env: swift.Swift) -> None:
        env.add(self._visual.body)
        self.update_visual()

    def state(self) -> np.ndarray:
        return np.asarray(self._robot.q, dtype=float)

    def update_visual(self) -> None:
        self._visual.update(state_to_pose(self.state()))

    def apply_velocity_cmd(self, command: Sequence[float]) -> None:
        # Here we directly use the step function of the unicycle model
        # because of how the unicycles are defined in roboticstoolbox
        self._robot.step(np.array(command, dtype=float))
        self.update_visual()

    def teleport(self, pose: Sequence[float]) -> None:
        self._robot.init(np.array(pose, dtype=float))
        self.update_visual()


In [17]:

# # Simulation parameters
# dt = 0.05
# simulation_time = 20.0  # seconds
# simulation_steps = int(simulation_time / dt)
world_bounds = (-2.5, 2.5, -2.5, 2.5)  # min_x, max_x, min_y, max_y

# Launch Swift viewer
env = swift.Swift()
env.launch(realtime=True, comms="rtc", browser="notebook")
time.sleep(1.0)



In [18]:
add_world_bounds(env, world_bounds)

cuboid,
[0.   0.   0.01]

In [22]:
# Create the unicycle model
robot = rtb.Unicycle()

# Initialize the robot state: (x, y, theta)
robot.init([0, 0, 0.15])  # Use .init() instead of assigning to .q

# Create a simple geometry (e.g., a box) to represent the unicycle
body_geom = sg.Cuboid(scale=[0.4, 0.2, 0.1], color=[0.2, 0.5, 0.8, 1.0])
# head_geom = sg.Cuboid(scale=[0.05, 0.05, 0.01], color=[0.8, 0.8, 0.8, 1.0])
env.add(body_geom)
# env.add(head_geom)


# make_agent(COLOR_BLUE)
# unicycle = [0.0, 0.0, 0.0]

3

In [ ]:
env.step(0.001)
# this is a test to see if connected  to guthub